# murcko scaffold

In [ ]:
import pandas as pd
df = pd.read_csv("../../data/processed/surechembl/molecular_descriptors_with_date.csv")

In [ ]:
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)

In [ ]:
from tqdm import tqdm
scaffolds = []
for i in tqdm(range(len(df))):
    smiles = df.iloc[i]["SMILES"]
    scaffold = get_scaffold(smiles)
    scaffolds.append(scaffold)

In [ ]:
import os
import sys

current_dir = os.getcwd()
parent_parent_dir = os.path.dirname(os.path.dirname(current_dir))
src_dir = os.path.join(parent_parent_dir, 'src')
sys.path.append(src_dir)

from util import *

In [ ]:
path = "../../data/processed/surechembl"
pickle_dump(scaffolds, os.path.join(path, "scaffolds.pkl"))

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def calc_props(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return {
            'Scaffold': smiles,
            'MW': Descriptors.MolWt(mol),            # 分子量
            'LogP': Descriptors.MolLogP(mol),        # 脂溶性
            'Fsp3': rdMolDescriptors.CalcFractionCSP3(mol), # 立体性(0-1)
            'H_Donors': Descriptors.NumHDonors(mol), # 水素結合供与体
            'RingCount': rdMolDescriptors.CalcNumRings(mol) # 環の数
        }
    except:
        return {
            'Scaffold': None,
            'MW': None,
            'LogP': None,
            'Fsp3': None,
            'H_Donors': None,
            'RingCount': None
        }

In [ ]:
result = []
for i in tqdm(range(len(scaffolds))):
    props = calc_props(scaffolds[i])
    if props:
        result.append(props)

In [ ]:
df_props = pd.DataFrame(result)
df_props.head()

df_props.to_csv("../../data/processed/surechembl/murcko_scaffold_properties_with_date.csv", index=False)

# BRICS scaffold

In [ ]:
from rdkit import Chem
from rdkit.Chem import BRICS
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

def get_brics_fragments(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    frag_mol = Chem.FragmentOnBRICSBonds(mol)
    frags = Chem.GetMolFrags(frag_mol, asMols=True)
    frag_smiles = set(Chem.MolToSmiles(f, True) for f in frags)
    return frag_smiles

In [ ]:
from tqdm import tqdm
import os

output_path = "../../data/processed/surechembl/molecular_descriptors_with_brics_fragments.csv"

if os.path.exists(output_path):
    with open(output_path, "r") as f:
        processed = sum(1 for _ in f) - 1  # header 分 -1
    if processed < 0:
        processed = 0
else:
    processed = 0

print(f"Resuming from index: {processed}")

with open(output_path, "a") as f:
    if processed == 0:
        f.write("SMILES,BRICS_Fragments\n")

    for i in tqdm(range(processed, len(df))):
        smiles = df["SMILES"].iloc[i]

        try:
            fragments = get_brics_fragments(smiles)
            frag_str = ";".join(fragments) if fragments else ""
        except Exception:
            print("!")
            frag_str = ""

        f.write(f"{smiles},\"{frag_str}\"\n")

In [ ]:
df = pd.read_csv(output_path)
df.head()

In [ ]:
import pandas as pd
import re
from tqdm import tqdm

for i in range(16):
    df[f"BRICS_{i}"] = 0

pattern = re.compile(r'\[(\d+)\*\]')

for idx, row in tqdm(df.iterrows(), total=len(df)):
    frags = row["BRICS_Fragments"]
    
    if pd.isna(frags):
        continue
    labels = pattern.findall(frags)
    
    counts = [0]*16
    for lbl in labels:
        counts[int(lbl)-1] += 1

    for i in range(16):
        df.at[idx, f"BRICS_{i}"] = counts[i]

In [ ]:
sc_smiles_date = pickle.load(open("../../data/processed/surechembl/250106_surechembl_smiles_date.pickle","rb"))

In [ ]:
df_filtered = df.loc[~(df[[f"BRICS_{i}" for i in range(16)]] == 0).all(axis=1)]

for i in tqdm(range(len(df_filtered))):
    smiles = df_filtered["SMILES"].iloc[i]
    if smiles in sc_smiles_date:
        df_filtered.at[i, "PATENT_DATE"] = sc_smiles_date[smiles]
    else:
        df_filtered.at[i, "PATENT_DATE"] = None

df_filtered.to_csv("molecular_descriptors_with_brics_counts_filtered.csv", index=False)